# Aula 4.1 — Carregar e exibir imagens com OpenCV
**Bloco 4 — OpenCV | Projeto IC: detecção de capacete em motociclistas por imagens de drone**

**Objetivo:** ler uma imagem de arquivo, entender o que o OpenCV devolve, corrigir o problema clássico das cores trocadas (BGR vs RGB) e salvar o resultado.

Conexão com os blocos anteriores: no Bloco 1 você viu que imagem é um array NumPy (`shape`, `dtype`, fatiamento). Aqui o OpenCV vai produzir esses arrays a partir de arquivos reais.

## 1. Tradução dos termos

| Termo | Tradução | O que faz / significa |
|---|---|---|
| **OpenCV** | Open Source Computer Vision Library (Biblioteca Aberta de Visão Computacional) | Conjunto de funções para ler, processar e desenhar em imagens |
| **cv2** | nome do módulo Python do OpenCV | É o que você importa: `import cv2` |
| **imread** | *image read* (ler imagem) | Carrega um arquivo de imagem e devolve um array NumPy |
| **imwrite** | *image write* (escrever imagem) | Salva um array NumPy em um arquivo de imagem |
| **imshow** | *image show* (mostrar imagem) | Abre uma janela com a imagem; **não funciona no Colab** |
| **cv2_imshow** | versão do `imshow` para o Colab | Mostra a imagem dentro da célula do notebook |
| **cvtColor** | *convert color* (converter cor) | Converte entre formatos de cor (ex.: BGR para RGB) |
| **BGR** | *Blue, Green, Red* (azul, verde, vermelho) | Ordem dos canais que o OpenCV usa |
| **RGB** | *Red, Green, Blue* (vermelho, verde, azul) | Ordem usada por Matplotlib, pelas telas e por quase todo o resto |
| **channel** | canal | Cada "camada" de cor da imagem (3 numa imagem colorida, 1 em cinza) |
| **grayscale** | escala de cinza | Imagem de 1 canal, com valores de 0 (preto) a 255 (branco) |
| **flag** | bandeira / opção | Parâmetro que escolhe um modo, como `cv2.IMREAD_GRAYSCALE` |
| **uint8** | *unsigned integer 8 bits* (inteiro sem sinal de 8 bits) | Tipo de dado dos pixels: valores de 0 a 255 |
| **None** | "nada" (valor vazio do Python) | O que `imread` devolve quando não consegue ler o arquivo |

## 2. Conceito

- **A imagem é um array NumPy.** `cv2.imread` devolve um array com shape `(altura, largura, 3)` e dtype `uint8`. Tudo o que você aprendeu de fatiamento e `shape` vale aqui.
- **O OpenCV usa a ordem BGR**, por razões históricas. Matplotlib e as telas usam RGB. Se você mostrar um array BGR direto no Matplotlib, as cores ficam trocadas: o **capacete vermelho aparece azul**. Para um projeto que pode usar cor como pista, isso é um erro silencioso perigoso.
- **A conversão é uma linha:** `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)`.
- **`imread` não lança erro quando falha.** Se o caminho estiver errado, ele devolve `None` (no máximo imprime um aviso no log) e o erro de verdade só aparece depois, em outra linha, com uma mensagem confusa. Por isso o projeto vai usar uma função segura de carregamento.
- **Indexação:** `img[y, x]` (linha primeiro, coluna depois). O pixel devolvido vem na ordem **B, G, R**.

## 3. Código aplicado ao projeto IC (Google Colab)

Para não depender de arquivos externos, a aula cria uma **imagem fictícia de vista de drone** com dois motociclistas: um **com capacete vermelho** (esquerda) e um **sem capacete** (direita). Depois você aprende a ler, exibir e salvar essa imagem. No fim há um bloco mostrando como usar as suas imagens reais.

### 3.1 Importar bibliotecas e criar a imagem de exemplo

In [ ]:
import cv2  # importa o OpenCV (o módulo se chama cv2)
import numpy as np  # NumPy: toda imagem no OpenCV é um array NumPy
import matplotlib.pyplot as plt  # Matplotlib: usado para exibir imagens no notebook
print("OpenCV versão:", cv2.__version__)  # mostra a versão instalada

def criar_cena_drone():  # monta uma imagem fictícia de "vista de drone" com dois motociclistas
    cena = np.full((240, 360, 3), 90, dtype=np.uint8)  # fundo cinza (asfalto): 240 linhas x 360 colunas x 3 canais
    cena[:, 175:185] = (0, 200, 255)  # faixa amarela no meio da pista (todas as linhas, colunas 175 a 184), em BGR
    cena[120:170, 70:90] = (40, 40, 40)  # moto da esquerda: retângulo escuro (valores em ordem BGR)
    cena[90:120, 72:88] = (60, 160, 60)  # corpo do piloto da esquerda: verde
    cena[70:90, 72:88] = (30, 30, 220)  # cabeça COM capacete vermelho (em BGR, vermelho = (30, 30, 220))
    cena[120:170, 250:270] = (40, 40, 40)  # moto da direita: retângulo escuro
    cena[90:120, 252:268] = (200, 90, 40)  # corpo do piloto da direita: azul
    cena[70:90, 252:268] = (120, 170, 220)  # cabeça SEM capacete (cor de pele em BGR)
    return cena  # devolve o array da cena

cena = criar_cena_drone()  # cria a imagem fictícia
cv2.imwrite("cena_drone.png", cena)  # salva em disco como PNG (formato sem perda de qualidade)
print("Arquivo cena_drone.png criado")  # confirma a criação

### 3.2 Ler a imagem com `cv2.imread` e inspecionar

In [ ]:
img = cv2.imread("cena_drone.png")  # lê o arquivo: devolve um array NumPy em ordem BGR
print("Tipo:", type(img))  # tipo do objeto devolvido: numpy.ndarray
print("Shape:", img.shape)  # (altura, largura, canais) = (240, 360, 3)
print("Dtype:", img.dtype)  # uint8: pixels de 0 a 255
print("Menor e maior valor:", img.min(), img.max())  # confirma a faixa de valores

### 3.3 O problema do BGR: o capacete vermelho fica azul

Vamos mostrar a mesma imagem de duas formas: direto (errado) e convertida para RGB (certo).

In [ ]:
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # converte de BGR (OpenCV) para RGB (Matplotlib)
fig, eixos = plt.subplots(1, 2, figsize=(10, 4))  # cria uma figura com 2 painéis lado a lado
eixos[0].imshow(img)  # painel 1: mostra o array BGR sem converter
eixos[0].set_title("ERRADO: BGR no Matplotlib")  # título do painel 1
eixos[1].imshow(img_rgb)  # painel 2: mostra o array já convertido para RGB
eixos[1].set_title("CERTO: RGB")  # título do painel 2
for e in eixos:  # percorre os dois painéis
    e.axis("off")  # esconde os números dos eixos
plt.show()  # exibe a figura

No painel da esquerda o capacete aparece **azul** e o piloto da direita fica **laranja**. No da direita, as cores estão corretas.

### 3.4 Ler um pixel: ordem B, G, R

In [ ]:
x, y = 80, 80  # coluna x e linha y no meio do capacete (atenção: x é coluna, y é linha)
b, g, r = img[y, x]  # o array é indexado [linha, coluna]; o pixel vem na ordem B, G, R
print("Pixel em BGR:", int(b), int(g), int(r))  # esperado: 30 30 220
print("Pixel em RGB:", int(r), int(g), int(b))  # reordenando para RGB: 220 30 30 (vermelho)

### 3.5 Ler em escala de cinza (1 canal)

In [ ]:
cinza = cv2.imread("cena_drone.png", cv2.IMREAD_GRAYSCALE)  # o flag pede a leitura já em escala de cinza
print("Shape em cinza:", cinza.shape)  # (240, 360): some a dimensão dos canais
plt.imshow(cinza, cmap="gray")  # com 1 canal é preciso pedir a paleta de cinza ao Matplotlib
plt.axis("off")  # esconde os eixos
plt.show()  # exibe a imagem

### 3.6 O perigo do `imread`: falha em silêncio

Se o caminho estiver errado, nenhuma exceção é lançada: o resultado é `None` (pode aparecer um aviso no log, mas o programa continua). Veja, e depois use uma função segura (vamos reutilizá-la no projeto).

In [ ]:
erro = cv2.imread("arquivo_que_nao_existe.png")  # caminho que não existe
print("Resultado:", erro)  # None: nenhuma exceção foi lançada

def carregar_imagem(caminho):  # função segura para carregar imagens do projeto
    imagem = cv2.imread(caminho)  # tenta ler o arquivo
    if imagem is None:  # se o OpenCV não conseguiu ler
        raise FileNotFoundError(f"Não consegui ler a imagem: {caminho}")  # lança um erro claro
    return imagem  # devolve o array BGR

try:  # testa a função segura com o caminho errado
    carregar_imagem("arquivo_que_nao_existe.png")  # deve lançar o erro
except FileNotFoundError as e:  # captura o erro esperado
    print("Erro claro:", e)  # mostra a mensagem

### 3.7 Salvar com `cv2.imwrite` (cuidado: ele espera BGR)

In [ ]:
sucesso = cv2.imwrite("cena_copia.png", img)  # salva o array BGR em um novo arquivo; devolve True se deu certo
print("Salvou?", sucesso)  # True se o arquivo foi gravado
recarregada = cv2.imread("cena_copia.png")  # lê o arquivo recém-salvo
print("Iguais?", np.array_equal(img, recarregada))  # PNG não perde qualidade: deve dar True

> **Regra de ouro:** `imread` e `imwrite` trabalham em **BGR**; Matplotlib trabalha em **RGB**. Converta só na hora de **mostrar** com Matplotlib. Se você converter para RGB e depois chamar `imwrite`, o arquivo salvo ficará com as cores trocadas.

### 3.8 Usando as suas imagens reais no Colab (não precisa rodar agora)

```python
from google.colab import files        # ferramenta de upload do Colab
enviados = files.upload()             # abre o seletor: escolha uma foto de drone
nome = list(enviados.keys())[0]       # pega o nome do arquivo enviado
img = carregar_imagem(nome)           # usa a função segura desta aula
```

Para mostrar com a função do Colab (em vez do Matplotlib):

```python
from google.colab.patches import cv2_imshow   # versão do imshow que funciona no Colab
cv2_imshow(img)                               # esta função aceita BGR direto, sem converter
```

## 4. Resumo

| Quero... | Uso |
|---|---|
| Ler um arquivo | `cv2.imread(caminho)` (devolve BGR, ou `None` se falhar) |
| Ler em cinza | `cv2.imread(caminho, cv2.IMREAD_GRAYSCALE)` |
| Mostrar com Matplotlib | `plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))` |
| Salvar | `cv2.imwrite(caminho, img)` (espera BGR) |
| Ver o pixel `(x, y)` | `img[y, x]` (vem em B, G, R) |

**Próximo passo:** faça o notebook de exercício `ex_4_1_carregar_exibir_imagens.ipynb`. A Aula 4.2 mostra como recortar, redimensionar e converter cores.